In [15]:
from datasets import load_dataset
data_files = {"train": "all_subsets/sampled_train_000000.tar"}

dataset = load_dataset("cubbk/hm_article_sorted_shards_dataset", data_files=data_files)

features_dataset = load_dataset("cubbk/hm_article_sorted_shards_dataset", data_files={"features": "all_subsets/features.json"})

print(dataset)

features = list(features_dataset["features"]["features"][0]) # type: ignore

features

DatasetDict({
    train: Dataset({
        features: ['cls', 'jpg', '__key__', '__url__'],
        num_rows: 105100
    })
})


['Accessories',
 'Bags',
 'Cosmetic',
 'Fun',
 'Furniture',
 'Garment Full body',
 'Garment Lower body',
 'Garment Upper body',
 'Garment and Shoe care',
 'Interior textile',
 'Items',
 'Nightwear',
 'Shoes',
 'Socks & Tights',
 'Stationery',
 'Swimwear',
 'Underwear',
 'Underwear/nightwear',
 'Unknown']

In [16]:
# split up training into training + validation
splits = dataset["train"].train_test_split(test_size=0.2, shuffle=True, seed=42) # type: ignore
train_ds = splits['train']
splits_test = splits['test'].train_test_split(test_size=0.5, shuffle=True, seed=42) # type: ignore

val_ds = splits_test['train']

confirm_last = splits_test['test']

# 632224001
# 806136004
# 902486003

In [18]:
from transformers import pipeline

pipe = pipeline("image-classification", model="cubbk/dinov2-base-finetuned-clothes-big")
pipe("https://static.zara.net/assets/public/bfee/addd/413e4fcc9fc2/f6d8e18c39c3/00653625600-e1/00653625600-e1.jpg?ts=1752067736279&w=1500")

Device set to use cuda:0


[{'label': 'Accessories', 'score': 0.9957412481307983},
 {'label': 'Garment Full body', 'score': 0.0015482051530852914},
 {'label': 'Garment Upper body', 'score': 0.0011444870615378022},
 {'label': 'Shoes', 'score': 0.00048461559345014393},
 {'label': 'Swimwear', 'score': 0.00044443688238970935}]

In [20]:
labels = features
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

id2label[1]

'Bags'

In [ ]:
# correct = 0
# total = len(confirm_last)
# for row in confirm_last:
#     correct_result = row["cls"] # type: ignore
#     correct_result_label = id2label[correct_result]
#     jpg = row["jpg"] # type: ignore
    
#     result = pipe(jpg)
#     best_result_label = result[0]["label"]
#     best_result_score = result[0]["score"]
#     print(best_result_label, best_result_score, correct_result_label)
#     correct += 1 if best_result_label == correct_result_label else 0

# print(str(correct) + "/" + str(total))


Underwear 0.999180257320404 Underwear
Garment Lower body 0.9848109483718872 Garment Lower body
Garment Upper body 0.9996252059936523 Garment Upper body
Garment Full body 0.9927855730056763 Garment Full body
Garment Upper body 0.955274224281311 Garment Upper body
Garment Full body 0.463260293006897 Garment Full body
Garment Full body 0.9894313812255859 Garment Full body
Garment Full body 0.9903762936592102 Garment Full body
Garment Upper body 0.9997050166130066 Garment Upper body
Accessories 0.999980092048645 Accessories
Accessories 0.9999102354049683 Accessories
Socks & Tights 0.9995506405830383 Socks & Tights
Garment Upper body 0.4714239835739136 Garment Upper body
Garment Upper body 0.9991206526756287 Garment Upper body
Garment Upper body 0.9976301193237305 Garment Upper body
Garment Lower body 0.9974023699760437 Garment Lower body
Garment Upper body 0.9998196959495544 Garment Upper body
Garment Upper body 0.9991047978401184 Garment Upper body


Garment Full body 0.6495022177696228 Garment Full body
Garment Lower body 0.9982155561447144 Garment Lower body
Garment Upper body 0.9820791482925415 Garment Upper body
Accessories 0.9983769655227661 Accessories
Garment Upper body 0.9934425354003906 Garment Upper body
Socks & Tights 0.9990467429161072 Socks & Tights
Garment Upper body 0.9987082481384277 Garment Upper body
Accessories 0.9880236387252808 Accessories
Garment Upper body 0.9996001124382019 Garment Upper body
Swimwear 0.989703357219696 Swimwear
Accessories 0.9978360533714294 Accessories
Nightwear 0.7535451054573059 Nightwear
Underwear 0.9993054866790771 Underwear
Garment Lower body 0.9907065033912659 Garment Lower body
Garment Upper body 0.9943053126335144 Garment Upper body
Shoes 0.9998075366020203 Shoes
Swimwear 0.9972054362297058 Swimwear
Accessories 0.9869846105575562 Accessories
Garment Lower body 0.999198853969574 Garment Lower body
Garment Full body 0.9687696099281311 Garment Upper body
Swimwear 0.6630741357803345 Swi

In [ ]:
# ...existing code...
from evaluate import evaluator

task = evaluator("image-classification")

# Map model label strings -> dataset class ids
label_mapping = {k: int(v) for k, v in getattr(pipe.model.config, "label2id", {}).items()}

results = task.compute(
    model_or_pipeline=pipe,
    data=confirm_last,
    input_column="jpg",
    label_column="cls",
    metric="accuracy",
    label_mapping=label_mapping,
)

print(results)

In [ ]:
# accuracy = (correct / total) * 100 if total else 0.0

# print(f"{correct}/{total} ({accuracy:.2f}%)")

10068/10510 (95.79%)
